### Loading important setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np


sys.path.append(str(Path("../src").resolve()))

In [2]:
from sentence_transformers import SentenceTransformer, CrossEncoder

model = SentenceTransformer("all-MiniLM-L6-v2")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

c:\Coding\Project Black Knight\project_black_knight\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7372.84it/s]


In [3]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../data/chroma")

movie_collection = chroma_client.get_collection("movies")
anime_collection = chroma_client.get_collection("animes")
music_collection = chroma_client.get_collection("music")
book_collection = chroma_client.get_collection("books")

print("Movies:", movie_collection.count())
print("Anime:", anime_collection.count())
print("Music:", music_collection.count())
print("Books:", book_collection.count())

Movies: 4805
Anime: 18336
Music: 32828
Books: 11230


In [4]:
movie_metadata_clean = pd.read_csv("../data/processed/movie_metadata_clean.csv")
anime_metadata_clean = pd.read_csv("../data/processed/anime_metadata_clean.csv")
music_metadata_clean = pd.read_csv("../data/processed/music_metadata_clean.csv")
book_metadata_clean = pd.read_csv("../data/processed/book_metadata_clean.csv")

# Ensure numeric types for filtering
book_metadata_clean["pages"] = pd.to_numeric(book_metadata_clean["pages"], errors="coerce")
book_metadata_clean["rating"] = pd.to_numeric(book_metadata_clean["rating"], errors="coerce")

print("Metadata loaded.")

print("Movies:", movie_metadata_clean.shape)
print("Anime:", anime_metadata_clean.shape)
print("Music:", music_metadata_clean.shape)
print("Books:", book_metadata_clean.shape)


Metadata loaded.
Movies: (4805, 9)
Anime: (18336, 7)
Music: (32828, 9)
Books: (11230, 10)


In [5]:
movie = pd.read_pickle("../data/processed/movies.pkl")
anime = pd.read_pickle("../data/processed/anime_tag2.pkl")
music = pd.read_pickle("../data/processed/songs_tag2.pkl")
book = pd.read_pickle("../data/processed/books_tag2.pkl")

In [6]:
movie_documents = movie["tag"].fillna("").astype(str).tolist()
anime_documents = anime["tag2"].fillna("").astype(str).tolist()
music_documents = music["tag"].fillna("").astype(str).tolist()
book_documents = book["tag2"].fillna("").astype(str).tolist()

movie_ids = [f"movie_{i}" for i in range(len(movie_documents))]
anime_ids = [f"anime_{i}" for i in range(len(anime_documents))]
music_ids = [f"music_{i}" for i in range(len(music_documents))]
book_ids = [f"book_{i}" for i in range(len(book_documents))]

print("Documents and IDs loaded.")

Documents and IDs loaded.


In [7]:
from rank_bm25 import BM25Okapi
import re


def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


movie_tokens = [tokenize(doc) for doc in movie_documents]
anime_tokens = [tokenize(doc) for doc in anime_documents]
music_tokens = [tokenize(doc) for doc in music_documents]
book_tokens = [tokenize(doc) for doc in book_documents]

movie_bm25 = BM25Okapi(movie_tokens)
anime_bm25 = BM25Okapi(anime_tokens)
music_bm25 = BM25Okapi(music_tokens)
book_bm25 = BM25Okapi(book_tokens)

bm25_indexes = {
    "movie": movie_bm25,
    "anime": anime_bm25,
    "music": music_bm25,
    "book": book_bm25
}

print("BM25 indexes loaded.")

BM25 indexes loaded.


In [9]:
from project_black_knight.query_parser import parse_query

In [10]:
print("Embedding model:", model)
print("Reranker loaded:", reranker is not None)

print("\nChroma:")
print("Movies:", movie_collection.count())
print("Anime:", anime_collection.count())
print("Music:", music_collection.count())
print("Books:", book_collection.count())

print("\nMetadata:")
print("Movies:", movie_metadata_clean.shape)
print("Anime:", anime_metadata_clean.shape)
print("Music:", music_metadata_clean.shape)
print("Books:", book_metadata_clean.shape)

print("\nBM25:")
print("Movies:", len(movie_tokens))
print("Anime:", len(anime_tokens))
print("Music:", len(music_tokens))
print("Books:", len(book_tokens))

print("\nQuery parser:", parse_query is not None)

Embedding model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
Reranker loaded: True

Chroma:
Movies: 4805
Anime: 18336
Music: 32828
Books: 11230

Metadata:
Movies: (4805, 9)
Anime: (18336, 7)
Music: (32828, 9)
Books: (11230, 10)

BM25:
Movies: 4806
Anime: 18336
Music: 32828
Books: 11230

Query parser: True


#### Semantic Retrieval

In [11]:
def semantic_retrieve(query, media_type, top_k=10):
    collections = {
        "movie": movie_collection,
        "anime": anime_collection,
        "music": music_collection,
        "book": book_collection
    }

    collection = collections[media_type]

    query_embedding = model.encode([query])

    result = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )

    return result

#### BM25 retrieval

In [12]:
def bm25_retrieve(query, media_type, top_k=10):
    bm25_index = bm25_indexes[media_type]

    ids = {
        "movie": movie_ids,
        "anime": anime_ids,
        "music": music_ids,
        "book": book_ids
    }[media_type]

    scores = bm25_index.get_scores(tokenize(query))

    top_indices = scores.argsort()[::-1][:top_k]

    return [
        {
            "id": ids[i],
            "score": float(scores[i])
        }
        for i in top_indices
    ]

#### RRF

In [13]:
def reciprocal_rank_fusion(result_lists, k=60):
    scores = {}

    for result_list in result_lists:
        for rank, item in enumerate(result_list, start=1):
            scores[item] = scores.get(item, 0) + 1 / (k + rank)

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

#### Hybrid Retrieval

In [14]:
import re

GENRE_MAP = {
    "sci-fi": ["sci-fi", "science fiction", "scifi", "science-fiction", "space"],
    "science fiction": ["sci-fi", "science fiction", "scifi", "science-fiction", "space"],
    "science-fiction": ["sci-fi", "science fiction", "scifi", "science-fiction", "space"],
    "scifi": ["sci-fi", "science fiction", "scifi", "science-fiction", "space"],
    "hard sci-fi": ["sci-fi", "science fiction", "scifi", "space", "fiction"],
    "electronic": ["edm", "electronic", "electropop", "electro", "dance", "synth", "pop"],
    "synthwave": ["edm", "electropop", "electro", "dance", "synth", "pop", "rock"],
    "dark fantasy": ["fantasy", "dark fantasy", "supernatural", "magic"],
}

def match_genre(requested_genre: str, row_genre_str: str) -> bool:
    norm_req = requested_genre.strip().lower()
    norm_row = row_genre_str.lower()
    for k, syns in GENRE_MAP.items():
        if k in norm_req:
            if any(s in norm_row for s in syns):
                return True
    if norm_req in norm_row:
        return True
    words = [w for w in re.findall(r"\b\w+\b", norm_req) if len(w) > 3]
    if any(w in norm_row for w in words):
        return True
    return False

def get_valid_ids(dataframe, current_ids, filters, media_type):
    valid_indices = []
    for idx, row in dataframe.iterrows():
        # Common: Release Year
        year = pd.to_numeric(row.get("release_year"), errors="coerce")
        if filters.get("release_year_min") is not None:
            if pd.isna(year) or year < filters["release_year_min"]:
                continue
        if filters.get("release_year_max") is not None:
            if pd.isna(year) or year > filters["release_year_max"]:
                continue

        # Common: Genres
        if media_type == "music":
            row_genres = f"{row.get('playlist_genre', '')} {row.get('playlist_subgenre', '')}".lower()
        else:
            row_genres = str(row.get("genres", "")).lower()

        if filters.get("genres_include"):
            if not any(match_genre(genre, row_genres) for genre in filters["genres_include"]):
                continue

        if filters.get("genres_exclude"):
            if any(match_genre(genre, row_genres) for genre in filters["genres_exclude"]):
                continue

        # Movie filters
        if media_type == "movie":
            runtime = pd.to_numeric(row.get("runtime"), errors="coerce")
            if filters.get("runtime_min") is not None:
                if pd.isna(runtime) or runtime < filters["runtime_min"]:
                    continue
            if filters.get("runtime_max") is not None:
                if pd.isna(runtime) or runtime > filters["runtime_max"]:
                    continue
            revenue = pd.to_numeric(row.get("revenue"), errors="coerce")
            if filters.get("revenue_min") is not None:
                if pd.isna(revenue) or revenue < filters["revenue_min"]:
                    continue

        # Anime filters
        if media_type == "anime":
            episodes = pd.to_numeric(row.get("episodes"), errors="coerce")
            if filters.get("episodes_min") is not None:
                if pd.isna(episodes) or episodes < filters["episodes_min"]:
                    continue
            if filters.get("episodes_max") is not None:
                if pd.isna(episodes) or episodes > filters["episodes_max"]:
                    continue
            popularity = pd.to_numeric(row.get("popularity"), errors="coerce")
            if filters.get("popularity_min") is not None:
                if pd.isna(popularity) or popularity < filters["popularity_min"]:
                    continue
            ranking = pd.to_numeric(row.get("ranking"), errors="coerce")
            if filters.get("ranking_max") is not None:
                if pd.isna(ranking) or ranking > filters["ranking_max"]:
                    continue

        # Music filters
        if media_type == "music":
            if filters.get("track_name") is not None:
                if str(row.get("track_name", "")).lower() != filters["track_name"].lower():
                    continue
            if filters.get("artist") is not None:
                if str(row.get("artist", "")).lower() != filters["artist"].lower():
                    continue
            if filters.get("album") is not None:
                if str(row.get("album", "")).lower() != filters["album"].lower():
                    continue
            if filters.get("playlist_genre") is not None:
                if str(row.get("playlist_genre", "")).lower() != filters["playlist_genre"].lower():
                    continue
            if filters.get("playlist_subgenre") is not None:
                if str(row.get("playlist_subgenre", "")).lower() != filters["playlist_subgenre"].lower():
                    continue
            duration = pd.to_numeric(row.get("duration_minutes"), errors="coerce")
            if filters.get("duration_min") is not None:
                if pd.isna(duration) or duration < filters["duration_min"]:
                    continue
            if filters.get("duration_max") is not None:
                if pd.isna(duration) or duration > filters["duration_max"]:
                    continue

        # Book filters
        if media_type == "book":
            rating = pd.to_numeric(row.get("rating"), errors="coerce")
            if filters.get("rating_min") is not None:
                if pd.isna(rating) or rating < filters["rating_min"]:
                    continue
            pages = pd.to_numeric(row.get("pages"), errors="coerce")
            if filters.get("pages_min") is not None:
                if pd.isna(pages) or pages < filters["pages_min"]:
                    continue
            if filters.get("pages_max") is not None:
                if pd.isna(pages) or pages > filters["pages_max"]:
                    continue
            if filters.get("author"):
                if not any(author.lower() in str(row.get("author", "")).lower() for author in filters["author"]):
                    continue
            if filters.get("publisher") is not None:
                if filters["publisher"].lower() not in str(row.get("publisher", "")).lower():
                    continue

        valid_indices.append(idx)

    return {current_ids[i] for i in valid_indices}


In [16]:
def search_black_knight(user_query: str, top_k: int = 10):
    # 1. LLM QUERY UNDERSTANDING
    parsed_query = parse_query(user_query)

    if not parsed_query.media_types:
        raise ValueError("No media type detected in query.")

    # 2. RETRIEVE FOR ALL IDENTIFIED MEDIA TYPES
    all_results = []
    common = parsed_query.common
    domain_queries = parsed_query.domain_queries
    semantic_query = parsed_query.semantic_query

    for media_type in parsed_query.media_types:
        domain_query = getattr(domain_queries, media_type, None)

        # BUILD FILTERS
        filters = {
            "genres_include": common.genres_include,
            "genres_exclude": common.genres_exclude,
            "release_year_min": common.release_year_min,
            "release_year_max": common.release_year_max,
        }

        if domain_query is not None:
            domain_filters = domain_query.model_dump(exclude_none=True)
            filters.update(domain_filters)

        # HYBRID RETRIEVAL FOR THIS MEDIA
        media_results = hybrid_retrieve(
            query=semantic_query,
            media_type=media_type,
            filters=filters,
            top_k=top_k
        )
        all_results.extend(media_results)

    # Sort combined results by rerank score
    all_results.sort(key=lambda x: x["score"], reverse=True)

    return {"parsed_query": parsed_query, "results": all_results[:top_k]}


In [17]:
def hybrid_retrieve(
    query,
    media_type,
    filters=None,
    top_k=10,
    candidate_k=30
):
    if filters is None:
        filters = {}

    collections = {
        "movie": movie_collection,
        "anime": anime_collection,
        "music": music_collection,
        "book": book_collection
    }

    bm25_indexes = {
        "movie": movie_bm25,
        "anime": anime_bm25,
        "music": music_bm25,
        "book": book_bm25
    }

    ids = {
        "movie": movie_ids,
        "anime": anime_ids,
        "music": music_ids,
        "book": book_ids
    }

    documents = {
        "movie": movie_documents,
        "anime": anime_documents,
        "music": music_documents,
        "book": book_documents
    }

    dataframes = {
        "movie": movie_metadata_clean,
        "anime": anime_metadata_clean,
        "music": music_metadata_clean,
        "book": book_metadata_clean
    }

    title_fields = {
        "movie": "title",
        "anime": "title",
        "music": "track_name",
        "book": "title"
    }

    if media_type not in collections:
        raise ValueError(
            f"Unsupported media type: {media_type}"
        )

    collection = collections[media_type]
    bm25_index = bm25_indexes[media_type]
    current_ids = ids[media_type]
    current_documents = documents[media_type]
    dataframe = dataframes[media_type]
    title_field = title_fields[media_type]

    id_to_index = {
        item_id: idx
        for idx, item_id in enumerate(current_ids)
    }


    # 1. METADATA FILTERING

    valid_ids = get_valid_ids(dataframe=dataframe, current_ids=current_ids, filters=filters, media_type=media_type)

    if not valid_ids:
        return []



    # 2. SEMANTIC RETRIEVAL

    query_embedding = model.encode([query])

    semantic_candidate_k = min(max(candidate_k * 5, 100), len(current_ids))

    semantic_result = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=semantic_candidate_k
    )

    semantic_ids_raw = semantic_result["ids"][0]

    semantic_ids = [
        item_id
        for item_id in semantic_ids_raw
        if item_id in valid_ids
    ]

    semantic_ids = semantic_ids[:candidate_k]



    # 3. BM25 RETRIEVAL

    query_tokens = tokenize(query)

    bm25_scores = bm25_index.get_scores(query_tokens)

    bm25_indices = bm25_scores.argsort()[::-1]

    bm25_ids = []

    for idx in bm25_indices:

        item_id = current_ids[idx]

        if item_id in valid_ids:
            bm25_ids.append(item_id)

        if len(bm25_ids) >= candidate_k:
            break



    # 4. RECIPROCAL RANK FUSION

    rrf_results = reciprocal_rank_fusion(
        [
            semantic_ids,
            bm25_ids
        ]
    )

    candidate_ids = [
        item_id
        for item_id, score in rrf_results[:candidate_k]
    ]

    if not candidate_ids:
        return []



    # 5. CROSS-ENCODER RERANKING

    candidate_documents = [
        current_documents[id_to_index[item_id]]
        for item_id in candidate_ids
    ]

    pairs = [
        [query, document]
        for document in candidate_documents
    ]

    rerank_scores = reranker.predict(pairs)

    reranked = sorted(
        zip(candidate_ids, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )


    # 6. FINAL RESULTS

    results = []

    for item_id, score in reranked[:top_k]:

        idx = id_to_index[item_id]

        results.append({
            "id": item_id,
            "title": dataframe.iloc[idx][title_field],
            "score": float(score),
            "media_type": media_type
        })

    return results

### LLM Add

In [18]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise RuntimeError("GROQ_API_KEY is not set")

client = Groq(api_key=api_key)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Groq connection works"
        }
    ]
)

print(response.choices[0].message.content)


Groq connection works


In [19]:
query = """
Recommend me mind-bending science fiction movies and anime,
released after 2010, with no horror.
I want movies under 150 minutes and anime with no more than 50 episodes.
Something similar in feel to Inception or Steins;Gate.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['movie', 'anime'] common=CommonQuery(title=None, genres_include=['science fiction'], genres_exclude=['horror'], release_year_min=2010, release_year_max=None) domain_queries=DomainQueries(movie=MovieQuery(production_companies=[], cast=[], director=[], revenue_min=None, runtime_min=None, runtime_max=150), anime=AnimeQuery(episodes_min=None, episodes_max=50, popularity_min=None, ranking_max=None), music=None, book=None) semantic_query='mind-bending' similar_to=['Inception', 'Steins;Gate']

========== RESULTS ==========
1. The Signal | movie | score=-6.3391
2. Listening | movie | score=-7.2481
3. Transcendence | movie | score=-7.3642
4. Cross Ange: Tenshi to Ryuu no Rondo | anime | score=-7.7169
5. Ingress the Animation | anime | score=-8.8572
6. Arve Rezzle: Kikaijikake no Yousei-tachi | anime | score=-9.1050
7. Accel World | anime | score=-9.5768
8. Kiznaiver | anime | score=-9.7408
9. Steins;Gate 0 | anime | score=-9.8608
10. Steins;Gate 

In [20]:
query = """
Recommend me mind-bending science fiction movies and anime,
released after 2010, with no horror.
I want movies under 150 minutes and anime with no more than 50 episodes.
Something similar in feel to Inception or Steins;Gate.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['movie', 'anime'] common=CommonQuery(title=None, genres_include=['science fiction'], genres_exclude=['horror'], release_year_min=2011, release_year_max=None) domain_queries=DomainQueries(movie=MovieQuery(production_companies=[], cast=[], director=[], revenue_min=None, runtime_min=None, runtime_max=150), anime=AnimeQuery(episodes_min=None, episodes_max=50, popularity_min=None, ranking_max=None), music=MusicQuery(track_name=None, artist=None, album=None, playlist_genre=None, playlist_subgenre=None, duration_min=None, duration_max=None), book=BookQuery(author=[], rating_min=None, isbn=None, characters=[], pages_min=None, pages_max=None, publisher=None)) semantic_query='mind-bending' similar_to=['Inception', 'Steins;Gate']

========== RESULTS ==========
1. The Signal | movie | score=-6.3391
2. Listening | movie | score=-7.2481
3. Transcendence | movie | score=-7.3642
4. Cross Ange: Tenshi to Ryuu no Rondo | anime | score=-7.7169
5. Ingress t

In [21]:
# INTERSTELLAR
query = """
Recommend me movies similar to a sci-fi film where an ex-NASA pilot leads a space mission through a wormhole near Saturn to find a new home for humanity, featuring planets with severe time dilation and a strong father-daughter bond. I want visually grand science fiction under 170 minutes released after 2010.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['movie'] common=CommonQuery(title=None, genres_include=['science-fiction'], genres_exclude=[], release_year_min=2011, release_year_max=None) domain_queries=DomainQueries(movie=MovieQuery(production_companies=[], cast=[], director=[], revenue_min=None, runtime_min=None, runtime_max=170), anime=None, music=None, book=None) semantic_query='visually grand science-fiction with an ex-NASA pilot, wormhole near Saturn, severe time dilation, and a strong father-daughter bond' similar_to=[]

========== RESULTS ==========
1. Interstellar | movie | score=-1.8843
2. Lockout | movie | score=-7.5975
3. The Martian | movie | score=-7.9868
4. Midnight Special | movie | score=-8.0158
5. Escape from Planet Earth | movie | score=-8.0684
6. Gravity | movie | score=-8.4380
7. Under the Skin | movie | score=-8.7262
8. Apollo 18 | movie | score=-9.1346
9. After Earth | movie | score=-9.1911
10. Guardians of the Galaxy | movie | score=-9.4099


In [22]:
# Avatar

query = """
Recommend me movies similar to an epic sci-fi story set in a distant alien moon's jungle, where a paralyzed marine uses a bio-engineered alien body to infiltrate a blue-skinned native tribe, only to turn against his corporate employers to protect the forest. I want visually immersive sci-fi action movies under 170 minutes.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['movie'] common=CommonQuery(title=None, genres_include=['sci-fi', 'action'], genres_exclude=[], release_year_min=None, release_year_max=None) domain_queries=DomainQueries(movie=MovieQuery(production_companies=[], cast=[], director=[], revenue_min=None, runtime_min=None, runtime_max=170), anime=None, music=None, book=None) semantic_query='visually immersive sci‑fi action in an alien moon jungle featuring a paralyzed marine using a bio‑engineered alien body to infiltrate blue‑skinned natives, turning against corporate employers to protect the forest' similar_to=[]

========== RESULTS ==========
1. U.F.O. | movie | score=-1.0960
2. Predator | movie | score=-1.3675
3. Avatar | movie | score=-2.0983
4. Predators | movie | score=-3.5407
5. I Am Number Four | movie | score=-4.1002
6. Aliens | movie | score=-4.3213
7. Independence Daysaster | movie | score=-4.5034
8. Species | movie | score=-4.8906
9. Independence Day: Resurgence | movie | score

In [23]:
# 1984

query = """
Recommend me novels similar to a sci-fi story about a wealthy industrialist who finances a deep-space expedition through an ancient alien stargate network to rescue a stranded crew, uncovering ancient cosmic mysteries along the way. I want hard sci-fi or portal-exploration books under 450 pages.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )

In [ ]:
# Alchemist

query = """
Recommend me novels similar to a short philosophical fable about a young shepherd boy who travels from Spain to the Egyptian pyramids in search of a hidden treasure, learning to listen to his heart and follow his personal destiny along the way. I want inspiring, quest-driven fiction under 250 pages.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['book'] common=CommonQuery(title=None, genres_include=[], genres_exclude=[], release_year_min=None, release_year_max=None) domain_queries=DomainQueries(movie=None, anime=None, music=None, book=BookQuery(author=[], rating_min=None, isbn=None, characters=[], pages_min=None, pages_max=250, publisher=None)) semantic_query='inspiring quest-driven philosophical fable' similar_to=[]

========== RESULTS ==========
1. True Dummy - a fable of existence | book | score=2.3362
2. Licence to Live: A Seeker's Journey to Greatness | book | score=-0.4905
3. World Peace: The Voice of a Mountain Bird | book | score=-3.4766
4. Old Turtle | book | score=-6.0183
5. The Blue Fox: A Novel | book | score=-6.4559
6. The Journey | book | score=-7.6769
7. The Journey Home | book | score=-8.0315
8. Phantastes | book | score=-8.7256
9. The Myth of Sisyphus | book | score=-8.8086
10. Conversations with God: An Uncommon Dialogue, Book 1 | book | score=-9.0473


In [ ]:
# Fullmetal Alchemist: Brotherhood

query = """
Recommend me anime similar to a story about two young brothers who lose their body parts in a failed, forbidden ritual to bring back their deceased mother, and go on a journey as military-backed alchemists to find a legendary stone that can restore their bodies. I want dark fantasy action anime with a complete plot under 70 episodes.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['anime'] common=CommonQuery(title=None, genres_include=['dark fantasy', 'action'], genres_exclude=[], release_year_min=None, release_year_max=None) domain_queries=DomainQueries(movie=None, anime=AnimeQuery(episodes_min=None, episodes_max=70, popularity_min=None, ranking_max=None), music=None, book=None) semantic_query='dark fantasy action anime with a complete plot under 70 episodes' similar_to=[]

========== RESULTS ==========
1. Persona 5 the Animation Recap | anime | score=1.3161
2. Inyouchuu Shoku: Ryoushokutou Taimaroku - Kanzenhan | anime | score=-0.4292
3. Inyouchuu Shoku: Ryoushokutou Taimaroku - Kanzenhan | anime | score=-0.4292
4. Gegege no Kitarou: Daikaijuu | anime | score=-1.0288
5. Supernatural The Animation | anime | score=-1.3347
6. Gegege no Kitarou: Nippon Bakuretsu | anime | score=-1.5608
7. Pokemon: Aki no Special | anime | score=-2.2602
8. Ar:pieL - 6-seosgae-ui Unmyeong | anime | score=-2.4051
9. Makai Senki Disgaea

In [ ]:
# Attack on Titan

query = """
Recommend me anime similar to a dark fantasy story about the last remnants of humanity living inside giant concentric walls to protect themselves from massive humanoid monsters, following a young soldier who vows to eradicate them after losing his home. I want intense, plot-twist-heavy mystery-action anime under 100 episodes.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['anime'] common=CommonQuery(title=None, genres_include=['Dark Fantasy', 'Mystery', 'Action'], genres_exclude=[], release_year_min=None, release_year_max=None) domain_queries=DomainQueries(movie=None, anime=AnimeQuery(episodes_min=None, episodes_max=100, popularity_min=None, ranking_max=None), music=None, book=None) semantic_query='intense, plot-twist-heavy mystery-action dark fantasy' similar_to=[]

========== RESULTS ==========
1. God Mazinger | anime | score=-2.9469
2. Dark Cat | anime | score=-3.0371
3. The Epic Of ZektBach | anime | score=-3.8819
4. Baccano! Specials | anime | score=-4.5144
5. Baccano! Specials | anime | score=-4.5144
6. Gegege no Kitarou: Daikaijuu | anime | score=-4.5826
7. Betterman | anime | score=-4.5891
8. Berserk | anime | score=-4.5943
9. Final Fantasy | anime | score=-4.9551
10. Gegege no Kitarou: Nippon Bakuretsu | anime | score=-5.3345


In [ ]:
# Daft Punk

query = """
Recommend me music tracks similar to a French electronic duo famous for wearing robot helmets, blending disco, house, and electronic synth rhythms into iconic dance tracks. I want energetic electronic instrumental tracks under 6 minutes.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['music'] common=CommonQuery(title=None, genres_include=['disco', 'house', 'electronic'], genres_exclude=[], release_year_min=None, release_year_max=None) domain_queries=DomainQueries(movie=None, anime=None, music=MusicQuery(track_name=None, artist=None, album=None, playlist_genre=None, playlist_subgenre=None, duration_min=None, duration_max=6.0), book=None) semantic_query='energetic electronic instrumental' similar_to=['Daft Punk']

========== RESULTS ==========
1. Energetic (Prequel Remix) | music | score=4.9608
2. Electricity | music | score=3.5298
3. On & On | music | score=3.0009
4. Shakalaka (Instrumental) | music | score=2.8952
5. On & On | music | score=2.8940
6. Electric Feel | music | score=2.7539
7. Electric Feel | music | score=2.2445
8. Looking Like This | music | score=2.1811
9. Kinetic (Original Mix) | music | score=2.1305
10. Realize My Dreams - Instrumental | music | score=2.0775


In [ ]:
# The Midnight

query = """
Recommend me music tracks similar to an 80s-inspired synthwave group that combines retro synth instruments, electronic basslines, and saxophones to create cinematic, nostalgic late-night soundscapes. I want atmospheric synthwave tracks released after 2015.
"""

result = search_black_knight(query, top_k=10)

print("\n========== PARSED QUERY ==========")
print(result["parsed_query"])

print("\n========== RESULTS ==========")

for i, item in enumerate(result["results"], start=1):
    print(
        f"{i}. {item['title']} "
        f"| {item['media_type']} "
        f"| score={item['score']:.4f}"
    )


========== PARSED QUERY ==========
media_types=['music'] common=CommonQuery(title=None, genres_include=['synthwave'], genres_exclude=[], release_year_min=2016, release_year_max=None) domain_queries=DomainQueries(movie=None, anime=None, music=MusicQuery(track_name=None, artist=None, album=None, playlist_genre='synthwave', playlist_subgenre=None, duration_min=None, duration_max=None), book=None) semantic_query='80s-inspired synthwave, retro synth instruments, electronic basslines, saxophones, cinematic, nostalgic late-night soundscapes, atmospheric' similar_to=[]

========== RESULTS ==========


In [ ]:
print(
    movie_metadata_clean[movie_metadata_clean["title"].str.contains("Interstellar",case=False,na=False)][
        ["title", "genres", "release_year", "runtime"]
    ]
)

           title                             genres  release_year  runtime
95  Interstellar  Adventure, Drama, Science Fiction          2014    169.0


In [ ]:
genres = movie_metadata_clean.loc[
    movie_metadata_clean["title"].str.lower() == "interstellar",
    "genres"
].iloc[0]

print(genres)
print(type(genres))
print("sci-fi" in str(genres).lower())
print("science fiction" in str(genres).lower())

Adventure, Drama, Science Fiction
<class 'str'>
False
True
